# Parameter diagnostics: variational dispersion + library centering experiments

Compare trained model parameters across 7 experiments:
- **Exp 1-4**: Single-modal RNA (regularise_bg x centering)
- **Exp 5-7**: Multimodal RNA+ATAC (centering variations)

All experiments use variational LogNormal dispersion posterior.

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams["pdf.fonttype"] = 42
rcParams["figure.figsize"] = 12, 4

## 1. Experiment configuration

In [ ]:
# Experiment definitions
experiments = {
    "exp1_rna_bg_noctr": {
        "type": "rna",
        "results_folder": "results/exp1_rna_bg_noctr/model",
        "label": "RNA: bg=T, ctr=OFF",
        "notebook": "model_comparisons/bone_marrow_gp_es_exp1_out.ipynb",
    },
    "exp2_rna_nobg_noctr": {
        "type": "rna",
        "results_folder": "results/exp2_rna_nobg_noctr/model",
        "label": "RNA: bg=F, ctr=OFF",
        "notebook": "model_comparisons/bone_marrow_gp_es_exp2_out.ipynb",
    },
    "exp3_rna_bg_ctr": {
        "type": "rna",
        "results_folder": "results/exp3_rna_bg_ctr/model",
        "label": "RNA: bg=T, ctr=ON",
        "notebook": "model_comparisons/bone_marrow_gp_es_exp3_out.ipynb",
    },
    "exp4_rna_nobg_ctr": {
        "type": "rna",
        "results_folder": "results/exp4_rna_nobg_ctr/model",
        "label": "RNA: bg=F, ctr=ON",
        "notebook": "model_comparisons/bone_marrow_gp_es_exp4_out.ipynb",
    },
    "exp5_mm_noctr": {
        "type": "multimodal",
        "results_folder": "results/exp5_mm_noctr/model",
        "label": "MM: ctr=OFF",
        "notebook": "bone_marrow_mm_es_exp5_out.ipynb",
    },
    "exp6_mm_ctr_rna": {
        "type": "multimodal",
        "results_folder": "results/exp6_mm_ctr_rna/model",
        "label": "MM: ctr=RNA",
        "notebook": "bone_marrow_mm_es_exp6_out.ipynb",
    },
    "exp7_mm_ctr_both": {
        "type": "multimodal",
        "results_folder": "results/exp7_mm_ctr_both/model",
        "label": "MM: ctr=RNA+ATAC",
        "notebook": "bone_marrow_mm_es_exp7_out.ipynb",
    },
    "exp8_mm_ctr_both_lowes": {
        "type": "multimodal",
        "results_folder": "results/exp8_mm_ctr_both_lowes/model",
        "label": "MM: ctr=both, lowES",
        "notebook": "bone_marrow_mm_es_exp8_out.ipynb",
    },
    "exp9_rna_nobg_ctr_lowes": {
        "type": "rna",
        "results_folder": "results/exp9_rna_nobg_ctr_lowes/model",
        "label": "RNA: ctr=ON, lowES",
        "notebook": "model_comparisons/bone_marrow_gp_es_exp9_out.ipynb",
    },
    "exp10a_mm_libvar_0.1": {
        "type": "multimodal",
        "results_folder": "results/exp10a_mm_libvar_0.1/model",
        "label": "MM: ctr=both, libvar=0.1",
        "notebook": "bone_marrow_mm_es_exp10a_out.ipynb",
    },
    "exp10b_mm_libvar_0.2": {
        "type": "multimodal",
        "results_folder": "results/exp10b_mm_libvar_0.2/model",
        "label": "MM: ctr=both, libvar=0.2",
        "notebook": "bone_marrow_mm_es_exp10b_out.ipynb",
    },
    "exp11_mm_learnable_scale": {
        "type": "multimodal",
        "results_folder": "results/exp11_mm_learnable_scale/model",
        "label": "MM: ctr=both, learnable scale",
        "notebook": "bone_marrow_mm_es_exp11_out.ipynb",
    },
    "exp12_rna_libvar_0.2": {
        "type": "rna",
        "results_folder": "results/exp12_rna_libvar_0.2/model",
        "label": "RNA: ctr=ON, libvar=0.2",
        "notebook": "model_comparisons/bone_marrow_gp_es_exp12_out.ipynb",
    },
    "exp8stratified": {
        "type": "multimodal",
        "results_folder": "results/exp8stratified/model",
        "label": "MM: ctr=both, stratified",
        "notebook": "bone_marrow_mm_es_exp8stratified_out.ipynb",
    },
    "exp9stratified": {
        "type": "rna",
        "results_folder": "results/exp9stratified/model",
        "label": "RNA: ctr=ON, stratified",
        "notebook": "model_comparisons/bone_marrow_gp_es_exp9stratified_out.ipynb",
    },
    "exp10bstratified": {
        "type": "multimodal",
        "results_folder": "results/exp10bstratified/model",
        "label": "MM: ctr=both, libvar=0.2, stratified",
        "notebook": "bone_marrow_mm_es_exp10bstratified_out.ipynb",
    },
    "exp11stratified": {
        "type": "multimodal",
        "results_folder": "results/exp11stratified/model",
        "label": "MM: ctr=both, learnable scale, stratified",
        "notebook": "bone_marrow_mm_es_exp11stratified_out.ipynb",
    },
    "exp12stratified": {
        "type": "rna",
        "results_folder": "results/exp12stratified/model",
        "label": "RNA: ctr=ON, libvar=0.2, stratified",
        "notebook": "model_comparisons/bone_marrow_gp_es_exp12stratified_out.ipynb",
    },
    "exp10bstratified_filtered": {
        "type": "multimodal",
        "results_folder": "results/exp10bstratified_filtered/model",
        "label": "MM: libvar=0.2, strat, ATAC>2k",
        "notebook": "bone_marrow_mm_es_exp10bstratified_filtered_out.ipynb",
    },
    "exp10bstratified_lowes": {
        "type": "multimodal",
        "results_folder": "results/exp10bstratified_lowes/model",
        "label": "MM: libvar=0.2, strat, lowES=1e-4",
        "notebook": "bone_marrow_mm_es_exp10bstratified_lowes_out.ipynb",
    },
    "exp11stratified_lowes": {
        "type": "multimodal",
        "results_folder": "results/exp11stratified_lowes/model",
        "label": "MM: learnable scale, strat, lowES=1e-4",
        "notebook": "bone_marrow_mm_es_exp11stratified_lowes_out.ipynb",
    },
    "exp10bstratified_filtered_lowes": {
        "type": "multimodal",
        "results_folder": "results/exp10bstratified_filtered_lowes/model",
        "label": "MM: libvar=0.2, strat, ATAC>2k, lowES",
        "notebook": "bone_marrow_mm_es_exp10bstratified_filtered_lowes_out.ipynb",
    },
    "exp10bstratified_filtered_lowes_libvar05": {
        "type": "multimodal",
        "results_folder": "results/exp10bstratified_filtered_lowes_libvar05/model",
        "label": "MM: libvar=0.5, strat, ATAC>2k, lowES",
        "notebook": "bone_marrow_mm_es_exp10bstratified_filtered_lowes_libvar05_out.ipynb",
    },
    "exp10bstratified_filtered_lowes2_libvar05": {
        "type": "multimodal",
        "results_folder": "results/exp10bstratified_filtered_lowes2_libvar05/model",
        "label": "MM: libvar=0.5, strat, ATAC>2k, ES=2e-4",
        "notebook": "bone_marrow_mm_es_exp10bstratified_filtered_lowes2_libvar05_out.ipynb",
    },
    "exp10bstratified_filtered_lowes_ataclr2": {
        "type": "multimodal",
        "results_folder": "results/exp10bstratified_filtered_lowes_ataclr2/model",
        "label": "MM: libvar=0.2, strat, ATAC>2k, lowES, ATAC-LR2x",
        "notebook": "bone_marrow_mm_es_exp10bstratified_filtered_lowes_ataclr2_out.ipynb",
    },
}

# Check which experiments have completed
for name, cfg in experiments.items():
    model_path = cfg["results_folder"]
    exists = os.path.exists(model_path)
    print(f"{'OK' if exists else 'MISSING':>7}  {name:30s}  {model_path}")

## 2. Load model state dicts

In [ ]:
# Load state dicts (lighter than full model load — no adata needed)
state_dicts = {}
for name, cfg in experiments.items():
    model_path = cfg["results_folder"]
    ckpt_path = os.path.join(model_path, "model.pt")
    if not os.path.exists(ckpt_path):
        print(f"SKIP {name}: {ckpt_path} not found")
        continue
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dicts[name] = ckpt.get("model_state_dict", ckpt)
    print(f"Loaded {name}: {len(state_dicts[name])} params")

print(f"\n{len(state_dicts)}/{len(experiments)} experiments loaded")

## 3. Parameter summary tables

In [ ]:
# Key parameters to inspect
rna_params = [
    "px_r_mu",
    "px_r_log_sigma",
    "dispersion_prior_rate_raw",
    "additive_background",
    "feature_scaling",
    "library_log_means",
    "library_log_vars",
]

mm_params = [
    "px_r_mu.rna",
    "px_r_mu.atac",
    "px_r_log_sigma.rna",
    "px_r_log_sigma.atac",
    "dispersion_prior_rate_raw.rna",
    "dispersion_prior_rate_raw.atac",
    "additive_background.rna",
    "feature_scaling.rna",
    "feature_scaling.atac",
    "library_log_means_rna",
    "library_log_means_atac",
    "library_log_vars_rna",
    "library_log_vars_atac",
]


def param_stats(sd, param_keys):
    """Extract summary stats for specified parameters."""
    rows = []
    for key in param_keys:
        if key in sd:
            t = sd[key].float()
            rows.append(
                {
                    "param": key,
                    "shape": str(list(t.shape)),
                    "mean": t.mean().item(),
                    "std": t.std().item() if t.numel() > 1 else 0.0,
                    "min": t.min().item(),
                    "max": t.max().item(),
                }
            )
    return pd.DataFrame(rows)


for name, sd in state_dicts.items():
    cfg = experiments[name]
    keys = mm_params if cfg["type"] == "multimodal" else rna_params
    df = param_stats(sd, keys)
    if len(df) > 0:
        print(f"\n{'=' * 80}")
        print(f"{cfg['label']} ({name})")
        print(f"{'=' * 80}")
        print(df.to_string(index=False, float_format="{:.4f}".format))
    else:
        print(f"\n{name}: no matching params found")

## 4. Cross-experiment comparison

In [ ]:
# Compare key scalar metrics across experiments
def extract_scalars(sd, exp_type):
    """Extract key scalar summaries from state dict."""
    from torch.nn.functional import softplus

    row = {}
    if exp_type == "rna":
        if "px_r_mu" in sd:
            theta = torch.exp(sd["px_r_mu"].float())
            row["theta_median"] = theta.median().item()
            row["theta_mean"] = theta.mean().item()
            row["theta_max"] = theta.max().item()
        if "px_r_log_sigma" in sd:
            sigma = torch.exp(sd["px_r_log_sigma"].float())
            row["theta_sigma_mean"] = sigma.mean().item()
        if "dispersion_prior_rate_raw" in sd:
            rate = softplus(sd["dispersion_prior_rate_raw"].float())
            row["disp_rate_mean"] = rate.mean().item()
        if "library_log_means" in sd:
            row["lib_log_mean"] = sd["library_log_means"].float().mean().item()
        if "additive_background" in sd:
            bg = torch.exp(sd["additive_background"].float())
            row["bg_mean"] = bg.mean().item()
            row["bg_max"] = bg.max().item()
    else:
        for mod in ["rna", "atac"]:
            if f"px_r_mu.{mod}" in sd:
                theta = torch.exp(sd[f"px_r_mu.{mod}"].float())
                row[f"theta_{mod}_median"] = theta.median().item()
                row[f"theta_{mod}_mean"] = theta.mean().item()
            if f"px_r_log_sigma.{mod}" in sd:
                sigma = torch.exp(sd[f"px_r_log_sigma.{mod}"].float())
                row[f"theta_sigma_{mod}_mean"] = sigma.mean().item()
            if f"dispersion_prior_rate_raw.{mod}" in sd:
                rate = softplus(sd[f"dispersion_prior_rate_raw.{mod}"].float())
                row[f"disp_rate_{mod}_mean"] = rate.mean().item()
            if f"library_log_means_{mod}" in sd:
                row[f"lib_log_mean_{mod}"] = sd[f"library_log_means_{mod}"].float().mean().item()

    return row


comp_rows = []
for name, sd in state_dicts.items():
    cfg = experiments[name]
    row = {"experiment": cfg["label"]}
    row.update(extract_scalars(sd, cfg["type"]))
    comp_rows.append(row)

df_comp = pd.DataFrame(comp_rows).set_index("experiment")
print("Cross-experiment comparison:")
print(df_comp.to_string(float_format="{:.4f}".format))

## 5. Dispersion (theta) distributions

In [ ]:
# Plot theta = exp(px_r_mu) distributions across RNA experiments
rna_exps = {k: v for k, v in state_dicts.items() if experiments[k]["type"] == "rna"}

if rna_exps:
    fig, axes = plt.subplots(1, len(rna_exps), figsize=(5 * len(rna_exps), 4), sharey=True)
    if len(rna_exps) == 1:
        axes = [axes]
    for ax, (name, sd) in zip(axes, rna_exps.items(), strict=False):
        if "px_r_mu" in sd:
            theta = torch.exp(sd["px_r_mu"].float()).flatten().numpy()
            ax.hist(theta, bins=50, alpha=0.7, color="steelblue")
            ax.axvline(np.median(theta), color="red", ls="--", label=f"median={np.median(theta):.1f}")
            ax.set_xlabel("theta = exp(px_r_mu)")
            ax.set_title(experiments[name]["label"])
            ax.legend(fontsize=8)
    axes[0].set_ylabel("Count")
    fig.suptitle("Learned dispersion (theta) — RNA experiments", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot theta distributions for multimodal experiments
mm_exps = {k: v for k, v in state_dicts.items() if experiments[k]["type"] == "multimodal"}

if mm_exps:
    fig, axes = plt.subplots(2, len(mm_exps), figsize=(5 * len(mm_exps), 8), sharey="row")
    if len(mm_exps) == 1:
        axes = axes.reshape(-1, 1)
    for j, (name, sd) in enumerate(mm_exps.items()):
        for i, mod in enumerate(["rna", "atac"]):
            key = f"px_r_mu.{mod}"
            if key in sd:
                theta = torch.exp(sd[key].float()).flatten().numpy()
                axes[i, j].hist(theta, bins=50, alpha=0.7, color="steelblue" if mod == "rna" else "darkorange")
                axes[i, j].axvline(np.median(theta), color="red", ls="--", label=f"median={np.median(theta):.1f}")
                axes[i, j].set_xlabel(f"theta_{mod}")
                axes[i, j].legend(fontsize=8)
            if i == 0:
                axes[i, j].set_title(experiments[name]["label"])
        axes[0, j].set_ylabel("RNA")
        axes[1, j].set_ylabel("ATAC")
    fig.suptitle("Learned dispersion (theta) — Multimodal experiments", fontsize=14)
    plt.tight_layout()
    plt.show()

## 6. Dispersion posterior uncertainty (sigma)

In [ ]:
# Scatter: theta_mu vs theta_sigma for each experiment
all_exps = list(state_dicts.items())
n = len(all_exps)
if n > 0:
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, (name, sd) in zip(axes, all_exps, strict=False):
        cfg = experiments[name]
        if cfg["type"] == "rna":
            mu_key, sig_key = "px_r_mu", "px_r_log_sigma"
        else:
            mu_key, sig_key = "px_r_mu.rna", "px_r_log_sigma.rna"
        if mu_key in sd and sig_key in sd:
            mu = sd[mu_key].float().flatten().numpy()
            sigma = torch.exp(sd[sig_key].float()).flatten().numpy()
            ax.scatter(mu, sigma, alpha=0.3, s=3)
            ax.set_xlabel("px_r_mu (log theta)")
            ax.set_title(cfg["label"], fontsize=10)
    axes[0].set_ylabel("px_r_sigma")
    fig.suptitle("Dispersion: mean vs posterior std", fontsize=14)
    plt.tight_layout()
    plt.show()

## 7. Library size prior comparison

In [ ]:
# Compare library_log_means across experiments
print("Library log-means per experiment:")
print(f"{'Experiment':<30s} {'mean':>8s} {'std':>8s} {'min':>8s} {'max':>8s}")
print("-" * 70)
for name, sd in state_dicts.items():
    cfg = experiments[name]
    if cfg["type"] == "rna":
        key = "library_log_means"
    else:
        key = "library_log_means_rna"
    if key in sd:
        t = sd[key].float()
        print(
            f"{cfg['label']:<30s} {t.mean().item():>8.4f} {t.std().item():>8.4f} {t.min().item():>8.4f} {t.max().item():>8.4f}"
        )

print()
print("Expected: centered experiments should have mean ~0, uncentered ~7.5")

## 8. Training history comparison

Load training history from output notebooks (if available) to compare ELBO convergence.

In [ ]:
# Load training history from model checkpoints
histories = {}
for name, cfg in experiments.items():
    model_path = cfg["results_folder"]
    attr_path = os.path.join(model_path, "attr.pkl")
    if os.path.exists(attr_path):
        import pickle

        with open(attr_path, "rb") as f:
            attrs = pickle.load(f)
        if "history_" in attrs:
            histories[name] = attrs["history_"]
            print(f"{name}: {len(attrs['history_'].get('elbo_train', []))} epochs")

if histories:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for name, hist in histories.items():
        label = experiments[name]["label"]
        if "elbo_train" in hist:
            train_elbo = hist["elbo_train"]["elbo_train"].values
            axes[0].plot(train_elbo, label=label, alpha=0.8)
        if "elbo_validation" in hist:
            val_elbo = hist["elbo_validation"]["elbo_validation"].values
            axes[1].plot(val_elbo, label=label, alpha=0.8)
    axes[0].set_title("Train ELBO")
    axes[1].set_title("Validation ELBO")
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.set_ylabel("ELBO")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No training histories found yet.")

## 9. Interpretation

Key questions to answer from the results above:

1. **Does variational px_r prevent theta explosion?**
   - Target: median theta in 5-50 range, not 400+ as seen with MAP optimisation.
   - Check the theta histograms (Section 5) and cross-experiment table (Section 4).
   - The posterior sigma (Section 6) should be non-negligible — if sigma collapses to ~0 the variational posterior is behaving like MAP.

2. **Does library centering speed up convergence?**
   - Compare training curves (Section 8) between centered (exp3/4) and uncentered (exp1/2) RNA experiments.
   - Centered experiments should converge in fewer epochs because the encoder starts near the prior (mean ~0 vs ~7.5).
   - Check final ELBO values — centering should not hurt final performance.

3. **Does regularise_background=False hurt or help?**
   - Compare exp1 (bg=T, ctr=OFF) vs exp2 (bg=F, ctr=OFF) and exp3 vs exp4 (same with centering ON).
   - With the previous Gamma(1,100) prior, background was crushed to ~0.00004 — effectively disabled.
   - If bg=False matches or beats bg=True, the background prior was not contributing.

4. **How do multimodal centering variants compare?**
   - exp5 (no centering) vs exp6 (RNA only) vs exp7 (RNA + ATAC).
   - Does centering RNA library help the joint latent?
   - Does additionally centering ATAC (sensitivity=0.2) help or hurt?

## 10. Per-cell library size: encoder posterior vs prior vs observed

Load selected RNA models with data to compare:
- **Observed** log-library size: `log(sum(counts))`
- **Prior** mean: `library_log_means[batch]` (per-batch, possibly centered)
- **Encoder posterior** mean: `ql_m` from the library encoder

If the prior variance is too tight (`library_log_vars_weight=0.05`), the encoder
may be forced to match the prior mean, leaving per-cell count variation unexplained
by library size — which the model then absorbs into dispersion (theta).

## 11. Summary: library prior variance vs early stopping sensitivity

Focused comparison of filtered/stratified multimodal experiments (ATAC>2k, stratified split):

| Experiment | libvar | ES delta | theta_rna | theta_atac | sigma_rna | sigma_atac |
|---|---|---|---|---|---|---|
| **Baseline** (default ES) | 0.2 | 3e-4 | 11.3 | 10.7 | 0.16 | 0.16 |
| lowES | 0.2 | 1e-4 | 12.1 | 11.3 | 0.19 | 0.19 |
| **libvar=0.5**, lowES | 0.5 | 1e-4 | 11.9 | 11.2 | 0.18 | 0.18 |
| **libvar=0.5, ES=2e-4** | 0.5 | 2e-4 | **11.7** | **11.0** | **0.17** | **0.17** |
| lowES + ATAC-LR 2x | 0.2 | 1e-4 | 13.8 | **17.4** | 0.24 | **0.57** |

**Key findings:**

1. **Wider library prior variance (libvar=0.5) reduces dispersion drift with low ES.** With lowES=1e-4, theta_rna drops from 12.1→11.9 and theta_atac from 11.3→11.2. The wider prior gives the encoder more freedom to explain per-cell library size variation, so dispersion doesn't need to absorb it.

2. **libvar=0.5 + ES=2e-4 is the sweet spot.** Theta values (11.7/11.0) nearly match the default-ES baseline (11.3/10.7), while allowing longer training. Sigma values (0.17) are close to baseline (0.16).

3. **ATAC-LR 2x is harmful.** It inflates theta_atac to 17.4 with sigma_atac 0.57 (3x baseline). The 2x ATAC learning rate over-trains the ATAC decoder, causing dispersion to absorb unwanted variation.

4. **ATAC>2k filtering helps modestly.** Comparing lowES with and without filtering: theta_rna 12.1 vs 12.5, theta_atac 11.3 vs 11.6.

In [ ]:
# Load and preprocess bone marrow RNA data (same pipeline as experiment notebooks)
import scanpy as sc
from regularizedvi.utils import download_bone_marrow_dataset, filter_genes

h5ad_path = download_bone_marrow_dataset(data_folder="data/")
adata = sc.read_h5ad(h5ad_path)
adata = adata[:, adata.var["feature_types"] == "GEX"].copy()
adata.var["SYMBOL"] = adata.var_names.values.copy()
adata.var_names = adata.var["gene_ids"].values.astype(str).copy()
adata.X = adata.layers["counts"]

# QC
adata.var["mt"] = [g.startswith("MT-") for g in adata.var["SYMBOL"]]
adata.obs["mt_frac"] = adata[:, adata.var["mt"].tolist()].X.sum(1).A.squeeze() / adata.obs["GEX_n_counts"]
selected_qc = filter_genes(adata, cell_count_cutoff=15, cell_percentage_cutoff2=0.05, nonz_mean_cutoff=1.07)
adata_scrub = adata[:, selected_qc].copy()
sc.pp.scrublet(adata_scrub, batch_key="batch", n_prin_comps=40, verbose=False)
adata.obs["doublet_score"] = adata_scrub.obs["doublet_score"]
del adata_scrub

adata = adata[
    (adata.obs["GEX_n_genes"] > 500)
    & (adata.obs["GEX_n_counts"] > 1000)
    & (adata.obs["GEX_n_counts"] < 80000)
    & (adata.obs["GEX_n_genes"] < 10000)
    & (adata.obs["ATAC_atac_fragments"] > 1000)
    & (adata.obs["ATAC_atac_fragments"] < 100000)
    & (adata.obs["mt_frac"] < 0.20)
    & (adata.obs["doublet_score"] < 0.18),
].copy()

# Gene selection
selected = filter_genes(adata, cell_count_cutoff=15, cell_percentage_cutoff2=0.01, nonz_mean_cutoff=1.03)
adata = adata[:, selected].copy()
adata.layers["counts"] = adata.X

# Observed log-library
observed_log_lib = np.log(np.array(adata.X.sum(1)).squeeze())
print(f"Data: {adata.shape}, observed log-lib: mean={observed_log_lib.mean():.3f}, std={observed_log_lib.std():.3f}")

In [ ]:
# Load selected RNA models and extract encoder library size posteriors
import regularizedvi

# Select key RNA experiments to compare
rna_experiments_to_load = ["exp1_rna_bg_noctr", "exp4_rna_nobg_ctr", "exp9_rna_nobg_ctr_lowes"]
rna_experiments_to_load = [e for e in rna_experiments_to_load if e in state_dicts]

library_results = {}
for exp_name in rna_experiments_to_load:
    cfg = experiments[exp_name]
    model_path = cfg["results_folder"]
    print(f"\nLoading {exp_name} from {model_path}...")

    # Setup anndata with same registration as training
    regularizedvi.AmbientRegularizedSCVI.setup_anndata(
        adata,
        layer="counts",
        ambient_covariate_keys=["batch"],
        nn_conditioning_covariate_keys=["site", "donor"],
        feature_scaling_covariate_keys=["site", "donor"],
        dispersion_key="batch",
        library_size_key="batch",
    )
    model = regularizedvi.AmbientRegularizedSCVI.load(model_path, adata=adata)

    # Run encoder on all cells to get library size posterior
    from scvi.dataloaders import AnnDataLoader

    dl = AnnDataLoader(model.adata_manager, batch_size=2048, shuffle=False)
    ql_means = []
    ql_vars = []
    prior_means = []
    prior_vars = []
    batch_indices = []

    device = next(model.module.parameters()).device
    model.module.eval()
    with torch.no_grad():
        for tensors in dl:
            # Move tensors to model device
            tensors = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in tensors.items()}
            inference_out = model.module.inference(**model.module._get_inference_input(tensors))
            ql = inference_out["ql"]
            ql_means.append(ql.loc.cpu())
            ql_vars.append(ql.scale.cpu() ** 2)

            # Get prior params for these cells
            gen_input = model.module._get_generative_input(tensors, inference_out)
            lib_idx = gen_input.get("library_size_index", gen_input.get("batch_index"))
            if lib_idx is None:
                lib_idx = tensors["batch"]
            local_means, local_vars = model.module._compute_local_library_params(lib_idx)
            prior_means.append(local_means.cpu())
            prior_vars.append(local_vars.cpu())
            batch_indices.append(tensors["batch"].cpu())

    library_results[exp_name] = {
        "ql_mean": torch.cat(ql_means).squeeze().numpy(),
        "ql_var": torch.cat(ql_vars).squeeze().numpy(),
        "prior_mean": torch.cat(prior_means).squeeze().numpy(),
        "prior_var": torch.cat(prior_vars).squeeze().numpy(),
        "batch": torch.cat(batch_indices).squeeze().numpy(),
        "label": cfg["label"],
    }
    print(
        f"  ql_mean: mean={library_results[exp_name]['ql_mean'].mean():.3f}, std={library_results[exp_name]['ql_mean'].std():.3f}"
    )
    print(
        f"  prior_mean: mean={library_results[exp_name]['prior_mean'].mean():.3f}, std={library_results[exp_name]['prior_mean'].std():.3f}"
    )
    print(f"  prior_var: mean={library_results[exp_name]['prior_var'].mean():.6f}")

print(f"\nLoaded {len(library_results)} models")

In [ ]:
# Plot: observed log-lib vs encoder posterior mean vs prior mean
n_exp = len(library_results)
if n_exp > 0:
    fig, axes = plt.subplots(2, n_exp, figsize=(6 * n_exp, 10))
    if n_exp == 1:
        axes = axes.reshape(-1, 1)

    for j, (_exp_name, res) in enumerate(library_results.items()):
        ql_m = res["ql_mean"]
        prior_m = res["prior_mean"]
        prior_v = res["prior_var"]

        # Row 1: observed vs encoder posterior mean
        ax = axes[0, j]
        ax.scatter(observed_log_lib, ql_m, alpha=0.05, s=1, c="steelblue", rasterized=True)
        ax.plot(
            [observed_log_lib.min(), observed_log_lib.max()],
            [observed_log_lib.min(), observed_log_lib.max()],
            "r--",
            lw=1,
            label="y=x",
        )
        # Show prior mean range as horizontal band
        prior_lo, prior_hi = prior_m.min(), prior_m.max()
        ax.axhspan(prior_lo, prior_hi, alpha=0.15, color="orange", label=f"prior mean [{prior_lo:.1f}, {prior_hi:.1f}]")
        ax.set_xlabel("Observed log-library")
        ax.set_ylabel("Encoder ql_mean")
        ax.set_title(res["label"])
        ax.legend(fontsize=7, loc="upper left")

        # Row 2: histogram of (ql_mean - prior_mean) and (observed - prior_mean)
        ax = axes[1, j]
        ax.hist(ql_m - prior_m, bins=100, alpha=0.6, color="steelblue", density=True, label="ql_mean - prior_mean")
        ax.hist(
            observed_log_lib - prior_m, bins=100, alpha=0.6, color="salmon", density=True, label="observed - prior_mean"
        )
        prior_std = np.sqrt(prior_v.mean())
        ax.axvline(0, color="black", ls="--", lw=1)
        ax.axvline(-prior_std, color="orange", ls=":", lw=1, label=f"prior std={prior_std:.4f}")
        ax.axvline(prior_std, color="orange", ls=":", lw=1)
        ax.set_xlabel("Deviation from prior mean")
        ax.set_ylabel("Density")
        ax.legend(fontsize=7)

    fig.suptitle("Library size: encoder posterior vs prior vs observed", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Summary table
    print(
        f"\n{'Experiment':<30s} {'obs_std':>8s} {'ql_std':>8s} {'prior_std':>10s} {'ql-obs corr':>12s} {'ql_range':>10s}"
    )
    print("-" * 85)
    for _exp_name, res in library_results.items():
        ql_m = res["ql_mean"]
        prior_v = res["prior_var"]
        corr = np.corrcoef(observed_log_lib, ql_m)[0, 1]
        print(
            f"{res['label']:<30s} {observed_log_lib.std():>8.4f} {ql_m.std():>8.4f} "
            f"{np.sqrt(prior_v.mean()):>10.4f} {corr:>12.4f} "
            f"[{ql_m.min():.2f}, {ql_m.max():.2f}]"
        )